# 🏭 Synthetic Production Data Generator
### 100% open-source — unique names every run, no API needed

**How names stay fresh every run:**
- Machine names built from combinatorial brand + model grammar (thousands of combinations)
- Product descriptions composed from rotating vocabularies with format variation
- Step names generated from operation + spec + dimension templates
- Seed auto-set from system time — never the same twice

| Section | What happens |
|---|---|
| **2** | ✏️ Set your counts — only cell to edit |
| **3** | Grammar vocabularies (edit to extend) |
| **4** | Generator functions |
| **5** | Run & generate |
| **6** | Validate & explore |
| **7** | Export JSON / CSV |


## 1. Imports

In [2]:
import json, random, time, re, string
import pandas as pd
from datetime import datetime
from itertools import product as iterproduct
from IPython.display import display, JSON as DJSON

# ── Auto-seed from current time — different every single run ──
SEED = int(time.time())
random.seed(SEED)
print(f"🎲 Seed: {SEED}  — changes automatically every run")
print("   (To reproduce a specific dataset, set SEED manually in this cell)")


🎲 Seed: 1780307050  — changes automatically every run
   (To reproduce a specific dataset, set SEED manually in this cell)


## 2. ✏️ Configuration — Only Cell You Need to Edit


In [3]:
# ═══════════════════════════════════════════════════════════
#  ✏️  EDIT THESE → then  Kernel → Restart & Run All
# ═══════════════════════════════════════════════════════════

NUM_PRODUCTS      = 12    # Products to generate      (5–30)
NUM_MACHINES      = 8     # Machines to generate      (4–15)
STEPS_PER_PRODUCT = 5     # Average steps per product (3–8)

# ── Demand distribution ──────────────────────────────────
DEMAND_WEIGHTS = [0.2, 0.5, 0.3]   # low / mid / high probability
DEMAND_RANGES  = {
    "low":  (80,   400),
    "mid":  (400,  1500),
    "high": (1500, 4500),
}
BATCH_SIZES = [50, 100, 150, 200, 250]

# ═══════════════════════════════════════════════════════════
print(f"✅  {NUM_PRODUCTS} products | {NUM_MACHINES} machines | ~{STEPS_PER_PRODUCT} steps/product")


✅  12 products | 8 machines | ~5 steps/product


## 3. Grammar Vocabularies
These lists are combined algorithmically to produce thousands of unique names.  
Add or change words here to shift the naming style.


In [4]:
# ═══════════════════════════════════════════════════════════
#  MACHINE GRAMMAR
#  Brand + Model-prefix + Model-number → "Vortex ZX-740"
# ═══════════════════════════════════════════════════════════

MACHINE_BRANDS = [
    "Vortex", "Nexus", "Helios", "Axon", "Praxis",
    "Vektor", "Orbis", "Ferrum", "Stratos", "Callix",
    "Dynex", "Kronos", "Lumis", "Torq", "Zephyr",
    "Cryon", "Valdex", "Pinnex", "Solux", "Arcus",
]

MACHINE_MODEL_PREFIXES = ["ZX", "TX", "MX", "CX", "RX", "EX", "SX", "AX", "DX", "FX"]
MACHINE_MODEL_NUMBERS  = list(range(200, 1000, 20))   # 200, 220, 240 ... 980

STATION_OPERATIONS = [
    "Wire Cutting", "Cable Stripping", "Crimping", "Conductor Joining",
    "Terminal Insertion", "Connector Assembly", "Wire Rolling",
    "Tape Wrapping", "Tube Assembly", "Grommet Fitting",
    "Seal Insertion", "Cable Bundling", "Continuity Testing",
    "Pull-Force Testing", "Visual Inspection",
]

MOLDING_MACHINE_BRANDS = [
    "Engel", "Battenfeld", "Demag", "Wittmann", "Haitian",
    "Chen Hsong", "Toyo", "Toshiba", "Niigata", "Sumitomo",
]
MOLDING_MODEL_PREFIXES = ["ST", "XT", "GT", "HT", "ET", "MT"]
MOLDING_MODEL_NUMBERS  = list(range(250, 800, 25))

# ═══════════════════════════════════════════════════════════
#  PRODUCT GRAMMAR
# ═══════════════════════════════════════════════════════════

WIRE_CONFIGS = [
    ("Single Wire",    "single",  1),
    ("2-Wire Jacket",  "jacket",  2),
    ("3-Wire Jacket",  "jacket",  3),
    ("4-Wire Jacket",  "jacket",  4),
    ("5-Wire Jacket",  "jacket",  5),
    ("6-Wire Jacket",  "jacket",  6),
    ("8-Wire Jacket",  "jacket",  8),
    ("10-Wire Jacket", "jacket", 10),
    ("Twisted Pair",   "twisted", 2),
    ("Twisted Quad",   "twisted", 4),
    ("Shielded 2-Wire","shielded",2),
    ("Shielded 4-Wire","shielded",4),
    ("Coaxial Cable",  "coaxial", 1),
]

CONNECTOR_FAMILIES = [
    ("DCC",   ["1x", "2x", "3x"]),
    ("MQS",   ["1x", "2x"]),
    ("JPT",   ["1x", "2x", "3x"]),
    ("HSD",   ["1x", "2x"]),
    ("FAKRA", ["1x", "2x"]),
    ("HVL",   ["1x"]),
    ("AMP",   ["1x", "2x", "3x"]),
    ("RAST",  ["1x", "2x"]),
    ("Micro-Fit", ["1x", "2x"]),
    ("Mini-Fit",  ["1x", "2x", "3x"]),
]

# Part code formats — each is a lambda that returns a random code
PART_CODE_FORMATS = [
    lambda: f"{random.choice(string.ascii_uppercase)}{random.randint(1,9)}{random.randint(1000,9999)}",
    lambda: f"{random.randint(10,99)}{random.choice('ABCDEFGHJKLMNPQRSTVWXYZ')}{random.randint(100,999)}",
    lambda: f"{''.join(random.choices(string.ascii_uppercase,k=2))}{random.randint(10000,99999)}",
    lambda: f"{random.choice(string.ascii_uppercase)}{random.randint(10,99)}-{random.randint(100,999)}",
    lambda: f"{random.randint(100,999)}-{random.choice(string.ascii_uppercase)}{random.randint(10,99)}",
    lambda: f"{''.join(random.choices(string.digits,k=3))}{''.join(random.choices(string.ascii_uppercase,k=2))}{random.randint(10,99)}",
]

VARIANT_SUFFIXES = [""] * 3 + list("ABCDEFGHJKMNPRST")  # blank weighted 3x

SUBVARIANTS = [
    "AA/AB/AC", "AD/AE/AF", "BA/BB/BC", "CA/CB/CC",
    "DA/DB", "EA/EB/EC", "FA/FB", "X1/X2/X3",
]

# ═══════════════════════════════════════════════════════════
#  PROCESS STEP GRAMMAR
#  operation + qualifier + spec → "Stripping & Crimping 4-Wire, 120mm"
# ═══════════════════════════════════════════════════════════

STEP_OPERATIONS = {
    "cut_strip": [
        "Cutting & Stripping",
        "Precision Cutting",
        "Stripping & End-Preparation",
        "Jacket Removal & Stripping",
        "Wire Cutting & Separation",
        "Multi-Wire Stripping",
        "Conductor Exposure",
    ],
    "crimp": [
        "Terminal Crimping",
        "Crimping & Sleeve Insertion",
        "Contact Crimping",
        "End-Crimp Application",
        "Wire Crimping & Assembly",
        "Seal Crimping",
        "Ferrule Crimping",
    ],
    "assembly": [
        "Connector Housing Assembly",
        "Terminal Insertion",
        "Connector Body Assembly",
        "Pin Insertion & Lock",
        "Connector Mating & Latching",
        "Seal & Connector Assembly",
        "Manual Connector Build",
        "Sub-Assembly Integration",
    ],
    "tube_grommet": [
        "Corrugated Tube Fitting",
        "PUR Tube Assembly",
        "Grommet Insertion",
        "Tube & Grommet Sub-Assembly",
        "Protective Sleeve Fitting",
        "Conduit Assembly",
        "Rubber Grommet Seating",
    ],
    "wrap_tape": [
        "Wire Pair Rolling & Taping",
        "Cable Bundling & Taping",
        "Spiral Wrap Application",
        "PVC Tape Wrapping",
        "Protective Taping",
        "Harness Taping",
        "Cloth Tape Application",
    ],
    "mold": [
        "Overmolding",
        "Injection Overmolding",
        "Connector Overmolding",
        "Strain-Relief Molding",
        "Encapsulation Molding",
        "Insert Molding",
    ],
    "test": [
        "Electrical Continuity Test",
        "Pull-Force Verification",
        "Visual Quality Inspection",
        "HV Withstand Test",
        "Seal Integrity Check",
    ],
}

STEP_QUALIFIERS = {
    "cut_strip": [
        "{n}-Wire Jacket, {l}mm",
        "Jacket Cable {n}-Wire, Strip Length {s}mm",
        "Single Conductors, {l}mm Cut Length",
        "{n}-Core Cable, {l}mm",
        "Twisted {n}-Wire, Strip {s}mm / Cut {l}mm",
    ],
    "crimp": [
        "{connector} Contact, Wire Gauge {g} AWG",
        "{connector} Terminal, Crimp Force {f}N",
        "Tin-Plated Contact, {g} AWG",
        "{connector} Seal Crimp, {g} AWG",
        "Double Crimp — Insulation + Conductor",
    ],
    "assembly": [
        "{connector} Housing, {p}-Pin",
        "{connector} Body, {p}-Way, Colour {col}",
        "{connector} Connector, CPA {cpa}",
        "Coding {coding} — {orient}",
        "{p}-Position {connector}, Locking Clip",
    ],
    "tube_grommet": [
        "{td}mm × {tw}mm Tube, {l}mm Length",
        "ID {td}mm PUR Tube, {l}–{l2}mm",
        "Rubber Grommet {td}mm, {l}mm Cable",
        "Corrugated Tube Ø{td}mm, {l}mm",
        "Split Tube {td}mm, {l}mm Section",
    ],
    "wrap_tape": [
        "Wire Pairs, {l}mm Overlap",
        "Full Harness, {t}mm Tape Width",
        "Spiral 50% Overlap, {l}mm Section",
        "{t}mm PVC Tape, {l}mm Bundle Length",
        "Cloth Tape, Double-Wrap {l}mm",
    ],
    "mold": [
        "{orient} Connector, {part}, {coding}, {cpa}",
        "{orient} Entry, {coding}, {cpa}",
        "Straight-Exit, {part}, {cpa}",
        "{orient} Bend, {coding}, CPA Clip {cpa}",
    ],
    "test": [
        "All Circuits, {v}V Continuity",
        "Crimp Pull-Force ≥{f}N, Sample {pct}%",
        "Visual — Seal & Terminal Seating",
        "{v}V HV Isolation, 1s Dwell",
    ],
}

# Filler values for qualifiers
STEP_FILLERS = {
    "n": [2, 3, 4, 5, 6, 7, 8, 10],
    "l": [80, 100, 110, 120, 150, 175, 200, 250, 300, 400, 500],
    "l2": [200, 250, 300, 400, 500, 600],
    "s": [5, 6, 7, 8, 10, 12, 15],
    "g": ["0.35", "0.5", "0.75", "1.0", "1.5", "2.5"],
    "f": [30, 40, 50, 60, 80, 100],
    "v": [12, 24, 48, 60],
    "p": [2, 3, 4, 6, 8, 12],
    "t": [9, 15, 19, 25],
    "td": ["3.5", "5.0", "6.0", "7.0", "8.0"],
    "tw": ["1.25", "1.35", "1.50", "1.75"],
    "col": ["Black", "Grey", "White", "Natural"],
    "cpa": ["With CPA", "No CPA"],
    "coding": ["Cod-A Black", "Cod-B White", "Cod-C Blue", "Cod-D Grey"],
    "orient": ["180° Straight", "90° Bottom", "90° Right", "45° Angled"],
    "part": ["85E-973-752", "3Q0-973-752", "4P0-973-752", "9J1-973-752", "95C-973-752"],
    "pct": [5, 10, 20],
    "connector": [],  # filled dynamically per product
}

print("✅ Grammar vocabularies loaded")
print(f"   Machine brand combinations: {len(MACHINE_BRANDS) * len(MACHINE_MODEL_PREFIXES) * len(MACHINE_MODEL_NUMBERS):,}")
print(f"   Wire × Connector combinations: {len(WIRE_CONFIGS) * sum(len(v) for _,v in CONNECTOR_FAMILIES):,}")
print(f"   Step operation × qualifier combinations: {sum(len(v) for v in STEP_OPERATIONS.values()) * sum(len(v) for v in STEP_QUALIFIERS.values()):,}")


✅ Grammar vocabularies loaded
   Machine brand combinations: 8,000
   Wire × Connector combinations: 299
   Step operation × qualifier combinations: 1,551


## 4. Generator Functions

In [5]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def rc(lst):
    """Random choice shorthand."""
    return random.choice(lst)

def fill_qualifier(template, connector="DCC"):
    """Fill {placeholders} in a step qualifier template."""
    STEP_FILLERS["connector"] = [connector]
    t = template
    for key, values in STEP_FILLERS.items():
        if f"{{{key}}}" in t and values:
            t = t.replace(f"{{{key}}}", str(rc(values)))
    # Fill any remaining with a sensible default
    t = re.sub(r"\{[^}]+\}", "", t)
    return t.strip().strip(",").strip()

def rand_cycle(low, high):
    return round(random.uniform(low, high), 2)

print("✅ Helpers ready")


✅ Helpers ready


In [6]:
# ── Machine Generator ────────────────────────────────────────────────────────

def generate_machines(n):
    """
    Builds n machines with fresh combinatorial names each run.
    Splits into 3 role buckets:
      - cutting/crimping pair machines  (~30%)
      - assembly stations               (~35%)
      - overmolding machines            (~35%)
    """
    machines = []
    used_names = set()

    def unique_name(candidates):
        for _ in range(50):
            name = rc(candidates)
            if name not in used_names:
                used_names.add(name)
                return name
        return candidates[0]  # fallback

    n_molding  = max(1, round(n * 0.30))
    n_cutting  = max(1, round(n * 0.30))
    n_stations = n - n_molding - n_cutting

    # Cutting / crimping pair machines
    for _ in range(n_cutting):
        b1, b2 = random.sample(MACHINE_BRANDS, 2)
        m1 = f"{rc(MACHINE_MODEL_PREFIXES)}-{rc(MACHINE_MODEL_NUMBERS)}"
        m2 = f"{rc(MACHINE_MODEL_PREFIXES)}-{rc(MACHINE_MODEL_NUMBERS)}"
        name = f"{b1} {m1} / {b2} {m2}"
        if name in used_names:
            m1 = f"{rc(MACHINE_MODEL_PREFIXES)}-{rc(MACHINE_MODEL_NUMBERS)}"
            name = f"{b1} {m1} / {b2} {m2}"
        used_names.add(name)
        machines.append({"name": name, "available_hours_per_day": 24.0, "_role": "cutting"})

    # Assembly stations
    ops = random.sample(STATION_OPERATIONS, min(n_stations, len(STATION_OPERATIONS)))
    if len(ops) < n_stations:
        ops += random.choices(STATION_OPERATIONS, k=n_stations - len(ops))
    for op in ops[:n_stations]:
        name = f"{op} Station"
        if name in used_names:
            name = f"{op} & QC Station"
        used_names.add(name)
        machines.append({"name": name, "available_hours_per_day": 16.0, "_role": "assembly"})

    # Overmolding / injection machines (grouped)
    brand = rc(MOLDING_MACHINE_BRANDS)
    model = f"{rc(MOLDING_MODEL_PREFIXES)}{rc(MOLDING_MODEL_NUMBERS)}"
    per_group = max(4, round(12 / max(n_molding, 1)))
    start = 1
    for _ in range(n_molding):
        end = start + per_group - 1
        name = f"{brand} {model} Machine {start}-{end}"
        used_names.add(name)
        machines.append({"name": name, "available_hours_per_day": 24.0, "_role": "molding"})
        start = end + 1
        # New brand/model for variety if more than one group
        brand = rc(MOLDING_MACHINE_BRANDS)
        model = f"{rc(MOLDING_MODEL_PREFIXES)}{rc(MOLDING_MODEL_NUMBERS)}"

    return machines


print("✅ Machine generator ready")


✅ Machine generator ready


In [7]:
# ── Product Generator ────────────────────────────────────────────────────────

def generate_products(n):
    """
    Generates n products with unique descriptions.
    Part codes are generated from rotating format lambdas so they
    look different every run (not always '9Y42xx' style).
    """
    products = []
    used_descs = set()

    for i in range(1, n + 1):
        # Wire config
        wire_label, wire_category, wire_count = rc(WIRE_CONFIGS)

        # Connector
        conn_family, conn_multipliers = rc(CONNECTOR_FAMILIES)
        conn_multiplier = rc(conn_multipliers)
        connector_str = f"{conn_multiplier}{conn_family}"
        dcc_type_map = {"1x": "Single", "2x": "Dual", "3x": "Triple"}
        dcc_type = f"{dcc_type_map.get(conn_multiplier, conn_multiplier)} {conn_family}"

        # Part code — pick a fresh format each time
        part_code_fn = rc(PART_CODE_FORMATS)
        part_code = part_code_fn()
        suffix = rc(VARIANT_SUFFIXES)

        # Subvariant for twisted/shielded
        subvariant = ""
        if wire_category in ("twisted", "shielded") and random.random() > 0.4:
            subvariant = f" {rc(SUBVARIANTS)}"

        description = f"{wire_label} {connector_str} Module {part_code}{suffix}{subvariant}"

        # Avoid duplicate descriptions
        if description in used_descs:
            part_code = part_code_fn()
            description = f"{wire_label} {connector_str} Module {part_code}{suffix}{subvariant}"
        used_descs.add(description)

        # Demand
        tier   = random.choices(["low", "mid", "high"], weights=DEMAND_WEIGHTS)[0]
        demand = random.randint(*DEMAND_RANGES[tier])
        batch  = rc(BATCH_SIZES)

        products.append({
            "item":         i,
            "sap_tn":       f"TN-{random.randint(100000, 999999)}",
            "sap_pl":       f"PL-{random.randint(1000, 9999)}" if random.random() > 0.25 else None,
            "dcc_type":     dcc_type,
            "description":  description,
            "demand_2024":  demand,
            "batch_size":   batch,
            "num_batches":  max(1, round(demand / batch)),
            # internal — stripped on export
            "_wire_category": wire_category,
            "_connector":     conn_family,
            "_wire_count":    wire_count,
        })

    return products


print("✅ Product generator ready")


✅ Product generator ready


In [8]:
# ── Process Step Generator ───────────────────────────────────────────────────

def build_step(op_category, product_connector, machine_name, step_num, product_item,
               cycle_range, workers):
    """Build one process step dict."""
    operation  = rc(STEP_OPERATIONS[op_category])
    qualifier_tmpl = rc(STEP_QUALIFIERS[op_category])
    qualifier  = fill_qualifier(qualifier_tmpl, connector=product_connector)
    step_name  = f"{operation} — {qualifier}" if qualifier else operation

    return {
        "product_item":       product_item,
        "step_number":        step_num,
        "machine_name":       machine_name,
        "step_name":          step_name,
        "cycle_time_seconds": rand_cycle(*cycle_range),
        "workers_required":   workers,
    }


def generate_process_steps(products, machines, avg_steps):
    """
    Assigns process steps to each product following factory flow:

        1. Cut & Strip       → cutting machine
        2. Crimp             → cutting machine (same or different)
        3. Tube / Grommet    → assembly station  [optional ~60%]
        4. Connector Assy    → assembly station
        5. Wrap / Tape       → assembly station  [optional ~40%]
        6. Overmolding       → molding machine   [always last]
        +  Test              → assembly station  [optional ~50%]

    Steps per product vary ±1 around avg_steps.
    """
    # Index machines by role
    cutting_m  = [m["name"] for m in machines if m.get("_role") == "cutting"]
    assembly_m = [m["name"] for m in machines if m.get("_role") == "assembly"]
    molding_m  = [m["name"] for m in machines if m.get("_role") == "molding"]

    # Fallbacks
    if not cutting_m:  cutting_m  = [machines[0]["name"]]
    if not assembly_m: assembly_m = [machines[1]["name"]]
    if not molding_m:  molding_m  = [machines[-1]["name"]]

    all_steps = []

    for product in products:
        item       = product["item"]
        connector  = product["_connector"]
        category   = product["_wire_category"]
        steps      = []
        step_num   = 1

        def add(op_cat, machine_pool, cycle_range, workers=0.5):
            nonlocal step_num
            steps.append(build_step(
                op_cat, connector, rc(machine_pool),
                step_num, item, cycle_range, workers
            ))
            step_num += 1

        # 1. Cut & Strip (always)
        add("cut_strip", cutting_m, (5.5, 9.5), workers=0.5)

        # 2. Crimp (always)
        add("crimp", cutting_m, (5.0, 9.0), workers=0.5)

        # 3. Tube / Grommet (jacket/shielded ~65%, twisted ~25%)
        tube_prob = 0.65 if category in ("jacket","shielded") else 0.25
        if random.random() < tube_prob:
            add("tube_grommet", assembly_m, (9.0, 20.0), workers=1.0)

        # 4. Connector Assembly (always)
        add("assembly", assembly_m, (7.0, 14.0), workers=1.0)

        # 5. Wrap / Tape (twisted ~80%, coaxial ~60%, others ~30%)
        wrap_probs = {"twisted": 0.80, "coaxial": 0.60, "jacket": 0.30, "shielded": 0.45, "single": 0.20}
        if random.random() < wrap_probs.get(category, 0.30):
            add("wrap_tape", assembly_m, (7.5, 13.0), workers=0.5)

        # 6. Optional extra assembly step to hit avg_steps target
        current = len(steps)
        target  = avg_steps + random.randint(-1, 1)
        if current < target - 1:   # room for one more before molding
            add("assembly", assembly_m, (8.0, 15.0), workers=1.0)

        # 7. Test step (50% chance)
        if random.random() < 0.50:
            add("test", assembly_m, (4.0, 8.0), workers=0.5)

        # 8. Overmolding — always last
        add("mold", molding_m, (13.0, 22.0), workers=1.0)

        all_steps.extend(steps)

    return all_steps


print("✅ Process step generator ready")


✅ Process step generator ready


## 5. Generate Data

In [9]:
# Re-seed here so Restart & Run All always gives fresh output
random.seed(int(time.time()))

print("⏳ Generating...")
machines      = generate_machines(NUM_MACHINES)
products      = generate_products(NUM_PRODUCTS)
process_steps = generate_process_steps(products, machines, STEPS_PER_PRODUCT)

# Build DataFrames for display/export
df_machines = pd.DataFrame([{k:v for k,v in m.items() if not k.startswith("_")} for m in machines])
df_products = pd.DataFrame([{k:v for k,v in p.items() if not k.startswith("_")} for p in products])
df_steps    = pd.DataFrame(process_steps)

print()
print("═" * 55)
print(f"  ✅ Machines:       {len(machines)}")
print(f"  ✅ Products:       {len(products)}")
print(f"  ✅ Process Steps:  {len(process_steps)}  (avg {len(process_steps)/len(products):.1f}/product)")
print("═" * 55)


⏳ Generating...

═══════════════════════════════════════════════════════
  ✅ Machines:       8
  ✅ Products:       12
  ✅ Process Steps:  74  (avg 6.2/product)
═══════════════════════════════════════════════════════


## 6. Explore & Validate

In [10]:
print("=== MACHINES ===")
display(df_machines)


=== MACHINES ===


,name,available_hours_per_day
0,Zephyr AX-940 / Pinnex RX-420,24.0
1,Kronos EX-580 / Zephyr FX-540,24.0
2,Wire Rolling Station,16.0
3,Visual Inspection Station,16.0
4,Pull-Force Testing Station,16.0
5,Terminal Insertion Station,16.0
6,Demag ET750 Machine 1-6,24.0
7,Wittmann HT350 Machine 7-12,24.0


In [11]:
print("=== PRODUCTS ===")
display(df_products[["item","description","dcc_type","demand_2024","batch_size","num_batches"]])


=== PRODUCTS ===


,item,description,dcc_type,demand_2024,batch_size,num_batches
0,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,1489,50,30
1,2,8-Wire Jacket 1xJPT Module U35625A,Single JPT,1325,250,5
2,3,Shielded 4-Wire 2xRAST Module 185-V87 CA/CB/CC,Dual RAST,1220,100,12
3,4,3-Wire Jacket 2xMini-Fit Module Y25-296C,Dual Mini-Fit,1049,100,10
4,5,Shielded 4-Wire 2xAMP Module 815-O79A CA/CB/CC,Dual AMP,3050,100,30
5,6,10-Wire Jacket 2xMicro-Fit Module 54H824,Dual Micro-Fit,530,250,2
6,7,Coaxial Cable 2xFAKRA Module Y34876N,Dual FAKRA,1692,200,8
7,8,Shielded 2-Wire 1xMQS Module 621OJ93E X1/X2/X3,Single MQS,205,150,1
8,9,5-Wire Jacket 1xHSD Module 494CD61,Single HSD,285,50,6
9,10,8-Wire Jacket 1xAMP Module 181-H37N,Single AMP,4058,200,20


In [12]:
print("=== PROCESS STEPS — first 20 rows ===")
display(df_steps.head(20))


=== PROCESS STEPS — first 20 rows ===


,product_item,step_number,machine_name,step_name,cycle_time_seconds,workers_required
0,1,1,Kronos EX-580 / Zephyr FX-540,"Wire Cutting & Separation — Single Conductors,...",8.40,0.5
1,1,2,Kronos EX-580 / Zephyr FX-540,"Seal Crimping — FAKRA Contact, Wire Gauge 1.5 AWG",5.94,0.5
2,1,3,Terminal Insertion Station,Manual Connector Build — Coding Cod-C Blue — 9...,9.62,1.0
3,1,4,Pull-Force Testing Station,"Cable Bundling & Taping — 15mm PVC Tape, 80mm ...",9.56,0.5
4,1,5,Pull-Force Testing Station,Pin Insertion & Lock — Coding Cod-D Grey — 90°...,11.90,1.0
5,1,6,Demag ET750 Machine 1-6,"Injection Overmolding — Straight-Exit, 3Q0-973...",15.64,1.0
6,2,1,Kronos EX-580 / Zephyr FX-540,"Conductor Exposure — 8-Core Cable, 100mm",7.31,0.5
7,2,2,Zephyr AX-940 / Pinnex RX-420,"Wire Crimping & Assembly — JPT Contact, Wire G...",5.72,0.5
8,2,3,Wire Rolling Station,"Conduit Assembly — 8.0mm × 1.75mm Tube, 150mm ...",14.07,1.0
9,2,4,Pull-Force Testing Station,"Pin Insertion & Lock — JPT Body, 2-Way, Colour...",12.72,1.0


In [13]:
print("=== STEP COUNT PER PRODUCT ===")
display(df_steps.groupby("product_item").size().rename("step_count").to_frame())


=== STEP COUNT PER PRODUCT ===


,step_count
product_item,
1,6
2,6
3,6
4,6
5,6
6,7
7,7
8,6
9,6


In [14]:
print("=== MACHINE WORKLOAD ===")
display(df_steps.groupby("machine_name").size().rename("steps_assigned")
        .sort_values(ascending=False).to_frame())


=== MACHINE WORKLOAD ===


,steps_assigned
machine_name,
Kronos EX-580 / Zephyr FX-540,14
Pull-Force Testing Station,13
Terminal Insertion Station,10
Visual Inspection Station,10
Zephyr AX-940 / Pinnex RX-420,10
Demag ET750 Machine 1-6,9
Wire Rolling Station,5
Wittmann HT350 Machine 7-12,3


In [15]:
print("=== ALL UNIQUE STEP NAMES ===")
for s in sorted(df_steps["step_name"].unique()):
    print(f"  • {s}")


=== ALL UNIQUE STEP NAMES ===
  • Cable Bundling & Taping — 15mm PVC Tape, 80mm Bundle Length
  • Cable Bundling & Taping — Cloth Tape, Double-Wrap 150mm
  • Conductor Exposure — 8-Core Cable, 100mm
  • Conductor Exposure — Single Conductors, 500mm Cut Length
  • Conduit Assembly — 8.0mm × 1.75mm Tube, 150mm Length
  • Conduit Assembly — Split Tube 7.0mm, 150mm Section
  • Connector Body Assembly — AMP Housing, 4-Pin
  • Connector Housing Assembly — HSD Connector, CPA No CPA
  • Connector Housing Assembly — RAST Body, 6-Way, Colour White
  • Connector Housing Assembly — RAST Connector, CPA No CPA
  • Connector Mating & Latching — 3-Position FAKRA, Locking Clip
  • Connector Mating & Latching — 6-Position Micro-Fit, Locking Clip
  • Connector Overmolding — 180° Straight Connector, 9J1-973-752, Cod-D Grey, With CPA
  • Contact Crimping — AMP Terminal, Crimp Force 60N
  • Contact Crimping — HSD Seal Crimp, 0.75 AWG
  • Cutting & Stripping — Single Conductors, 250mm Cut Length
  • Cutting 

In [16]:
print("=== FULL PRODUCT → STEP TRACE ===")
trace = df_steps.merge(df_products[["item","description"]], left_on="product_item", right_on="item")
display(trace[["item","description","step_number","step_name","machine_name","cycle_time_seconds"]])


=== FULL PRODUCT → STEP TRACE ===


,item,description,step_number,step_name,machine_name,cycle_time_seconds
0,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,1,"Wire Cutting & Separation — Single Conductors,...",Kronos EX-580 / Zephyr FX-540,8.40
1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,2,"Seal Crimping — FAKRA Contact, Wire Gauge 1.5 AWG",Kronos EX-580 / Zephyr FX-540,5.94
2,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,3,Manual Connector Build — Coding Cod-C Blue — 9...,Terminal Insertion Station,9.62
3,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,4,"Cable Bundling & Taping — 15mm PVC Tape, 80mm ...",Pull-Force Testing Station,9.56
4,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,5,Pin Insertion & Lock — Coding Cod-D Grey — 90°...,Pull-Force Testing Station,11.90
...,...,...,...,...,...,...
69,12,2-Wire Jacket 1xMini-Fit Module 181GO44D,2,"Seal Crimping — Tin-Plated Contact, 0.75 AWG",Zephyr AX-940 / Pinnex RX-420,8.08
70,12,2-Wire Jacket 1xMini-Fit Module 181GO44D,3,Protective Sleeve Fitting — 7.0mm × 1.75mm Tub...,Visual Inspection Station,9.87
71,12,2-Wire Jacket 1xMini-Fit Module 181GO44D,4,"Pin Insertion & Lock — Mini-Fit Body, 8-Way, C...",Visual Inspection Station,10.11
72,12,2-Wire Jacket 1xMini-Fit Module 181GO44D,5,Visual Quality Inspection — Visual — Seal & Te...,Wire Rolling Station,7.67


## 7. Export

In [17]:
def clean(lst):
    return [{k:v for k,v in d.items() if not k.startswith("_")} for d in lst]

# JSON
output = {
    "generated_at": datetime.now().isoformat(),
    "seed": SEED,
    "machines":      clean(machines),
    "products":      clean(products),
    "process_steps": process_steps,
}
with open("synthetic_production_data.json","w") as f:
    json.dump(output, f, indent=2)

# CSV
df_machines.to_csv("machines.csv", index=False)
df_products.to_csv("products.csv", index=False)
df_steps.to_csv("process_steps.csv", index=False)

print("Machines")
display(df_machines)
print("Products")
display(df_products)
print("Process Steps")
display(df_steps)

print("✅ Exported:")
print("   synthetic_production_data.json")
print(f"   machines.csv       ({len(machines)} rows)")
print(f"   products.csv       ({len(products)} rows)")
print(f"   process_steps.csv  ({len(process_steps)} rows)")


Machines


,name,available_hours_per_day
0,Zephyr AX-940 / Pinnex RX-420,24.0
1,Kronos EX-580 / Zephyr FX-540,24.0
2,Wire Rolling Station,16.0
3,Visual Inspection Station,16.0
4,Pull-Force Testing Station,16.0
5,Terminal Insertion Station,16.0
6,Demag ET750 Machine 1-6,24.0
7,Wittmann HT350 Machine 7-12,24.0


Products


,item,sap_tn,sap_pl,dcc_type,description,demand_2024,batch_size,num_batches
0,1,TN-651165,PL-8398,Dual FAKRA,Twisted Pair 2xFAKRA Module 32V845D FA/FB,1489,50,30
1,2,TN-878056,PL-9360,Single JPT,8-Wire Jacket 1xJPT Module U35625A,1325,250,5
2,3,TN-714829,PL-4168,Dual RAST,Shielded 4-Wire 2xRAST Module 185-V87 CA/CB/CC,1220,100,12
3,4,TN-830142,PL-9785,Dual Mini-Fit,3-Wire Jacket 2xMini-Fit Module Y25-296C,1049,100,10
4,5,TN-626398,PL-2957,Dual AMP,Shielded 4-Wire 2xAMP Module 815-O79A CA/CB/CC,3050,100,30
5,6,TN-615842,PL-1549,Dual Micro-Fit,10-Wire Jacket 2xMicro-Fit Module 54H824,530,250,2
6,7,TN-842764,None,Dual FAKRA,Coaxial Cable 2xFAKRA Module Y34876N,1692,200,8
7,8,TN-456590,None,Single MQS,Shielded 2-Wire 1xMQS Module 621OJ93E X1/X2/X3,205,150,1
8,9,TN-377112,PL-8079,Single HSD,5-Wire Jacket 1xHSD Module 494CD61,285,50,6
9,10,TN-149462,PL-4192,Single AMP,8-Wire Jacket 1xAMP Module 181-H37N,4058,200,20


Process Steps


,product_item,step_number,machine_name,step_name,cycle_time_seconds,workers_required
0,1,1,Kronos EX-580 / Zephyr FX-540,"Wire Cutting & Separation — Single Conductors,...",8.40,0.5
1,1,2,Kronos EX-580 / Zephyr FX-540,"Seal Crimping — FAKRA Contact, Wire Gauge 1.5 AWG",5.94,0.5
2,1,3,Terminal Insertion Station,Manual Connector Build — Coding Cod-C Blue — 9...,9.62,1.0
3,1,4,Pull-Force Testing Station,"Cable Bundling & Taping — 15mm PVC Tape, 80mm ...",9.56,0.5
4,1,5,Pull-Force Testing Station,Pin Insertion & Lock — Coding Cod-D Grey — 90°...,11.90,1.0
...,...,...,...,...,...,...
69,12,2,Zephyr AX-940 / Pinnex RX-420,"Seal Crimping — Tin-Plated Contact, 0.75 AWG",8.08,0.5
70,12,3,Visual Inspection Station,Protective Sleeve Fitting — 7.0mm × 1.75mm Tub...,9.87,1.0
71,12,4,Visual Inspection Station,"Pin Insertion & Lock — Mini-Fit Body, 8-Way, C...",10.11,1.0
72,12,5,Wire Rolling Station,Visual Quality Inspection — Visual — Seal & Te...,7.67,0.5


✅ Exported:
   synthetic_production_data.json
   machines.csv       (8 rows)
   products.csv       (12 rows)
   process_steps.csv  (74 rows)


In [18]:
# Django fixtures (optional — replace 'yourapp' with your app label)
APP = "yourapp"
fixtures = []
for m in clean(machines):
    fixtures.append({"model": f"{APP}.machine", "fields": m})
for p in clean(products):
    fixtures.append({"model": f"{APP}.product", "pk": p["item"],
                     "fields": {k:v for k,v in p.items() if k != "item"}})
for s in process_steps:
    fixtures.append({"model": f"{APP}.processstep", "fields": s})

with open("fixtures.json","w") as f:
    json.dump(fixtures, f, indent=2)

print(f"✅ Django fixtures → fixtures.json  ({len(fixtures)} entries)")
print(f"   Run: python manage.py loaddata fixtures.json")


✅ Django fixtures → fixtures.json  (94 entries)
   Run: python manage.py loaddata fixtures.json


## 8. Raw JSON Preview — First Product

In [19]:
first = clean(products)[0]
first_steps = [s for s in process_steps if s["product_item"] == first["item"]]
display(DJSON({"product": first, "process_steps": first_steps}))


<IPython.core.display.JSON object>

## 9. Batch Scheduling and Gantt Chart

This section combines products, process steps, and machine availability into a batch-level production schedule. It enforces machine capacity, step order, shift windows, duration consistency, and batch coherence.


In [20]:
from datetime import datetime, timedelta, time as dtime
import math

SCHEDULE_START = pd.Timestamp("2026-06-01 08:00")
DEFAULT_SHIFT_START_HOUR = 8


def machine_shift_windows(machine_row):
    """Return allowed shift windows as (start_hour, end_hour) for one machine."""
    hours = float(machine_row.get("available_hours_per_day", 24.0))
    if hours >= 24:
        return [(0, 24)]
    if hours == 16:
        return [(8, 16), (16, 24)]
    if hours == 8:
        return [(8, 16)]
    start = DEFAULT_SHIFT_START_HOUR
    end = min(24, start + int(hours))
    return [(start, end)]


def _day_start(ts):
    return pd.Timestamp(ts).normalize()


def next_shift_start(ts, windows):
    """Move timestamp to the next valid shift start for the given windows."""
    ts = pd.Timestamp(ts)
    for day_offset in range(0, 14):
        day = _day_start(ts) + pd.Timedelta(days=day_offset)
        for start_h, end_h in windows:
            shift_start = day + pd.Timedelta(hours=start_h)
            shift_end = day + pd.Timedelta(hours=end_h)
            if ts <= shift_start:
                return shift_start
            if shift_start <= ts < shift_end:
                return ts
    raise RuntimeError("Could not find a valid shift start")


def fit_inside_shift(earliest_start, duration, windows):
    """Find the earliest start/end pair where the whole operation fits inside one shift."""
    candidate = pd.Timestamp(earliest_start)
    for _ in range(10000):
        candidate = next_shift_start(candidate, windows)
        day = _day_start(candidate)
        for start_h, end_h in windows:
            shift_start = day + pd.Timedelta(hours=start_h)
            shift_end = day + pd.Timedelta(hours=end_h)
            if shift_start <= candidate < shift_end and candidate + duration <= shift_end:
                return candidate, candidate + duration
        candidate = _day_start(candidate) + pd.Timedelta(days=1, hours=windows[0][0])
    raise RuntimeError("Could not fit operation inside shift windows")


def find_machine_slot(machine_ready_at, earliest_start, duration, windows):
    """Earliest feasible slot after both the batch predecessor and machine are ready."""
    candidate = max(pd.Timestamp(machine_ready_at), pd.Timestamp(earliest_start))
    while True:
        start, end = fit_inside_shift(candidate, duration, windows)
        if start >= pd.Timestamp(machine_ready_at):
            return start, end
        candidate = pd.Timestamp(machine_ready_at)


def build_batch_schedule(df_products, df_steps, df_machines, schedule_start=SCHEDULE_START):
    products_i = df_products.copy()
    steps_i = df_steps.copy().sort_values(["product_item", "step_number"])
    machines_i = df_machines.copy()

    machine_windows = {
        row["name"]: machine_shift_windows(row)
        for _, row in machines_i.iterrows()
    }
    machine_ready_at = {name: pd.Timestamp(schedule_start) for name in machines_i["name"]}
    rows = []

    for _, product in products_i.sort_values("item").iterrows():
        product_steps = steps_i[steps_i["product_item"] == product["item"]].sort_values("step_number")
        for batch_num in range(1, int(product["num_batches"]) + 1):
            batch_id = f"P{int(product['item']):03d}-B{batch_num:03d}"
            batch_ready_at = pd.Timestamp(schedule_start)
            for _, step in product_steps.iterrows():
                machine_name = step["machine_name"]
                duration_hours = (float(step["cycle_time_seconds"]) * float(product["batch_size"])) / 3600.0
                duration = pd.Timedelta(hours=duration_hours)
                start_time, end_time = find_machine_slot(
                    machine_ready_at[machine_name],
                    batch_ready_at,
                    duration,
                    machine_windows[machine_name],
                )
                rows.append({
                    "batch_id": batch_id,
                    "batch_num": batch_num,
                    "product_item": product["item"],
                    "product_description": product["description"],
                    "dcc_type": product["dcc_type"],
                    "batch_size": product["batch_size"],
                    "step_number": step["step_number"],
                    "step_name": step["step_name"],
                    "machine_name": machine_name,
                    "cycle_time_seconds": step["cycle_time_seconds"],
                    "workers_required": step["workers_required"],
                    "start_time": start_time,
                    "end_time": end_time,
                    "duration_hours": round(duration_hours, 6),
                })
                machine_ready_at[machine_name] = end_time
                batch_ready_at = end_time

    schedule = pd.DataFrame(rows)
    schedule["start_date"] = schedule["start_time"].dt.date
    schedule["end_date"] = schedule["end_time"].dt.date
    schedule["product_batch"] = schedule["batch_id"] + " | " + schedule["product_description"].str.slice(0, 45)
    return schedule


def validate_schedule(schedule, df_machines):
    errors = []
    machine_windows = {
        row["name"]: machine_shift_windows(row)
        for _, row in df_machines.iterrows()
    }

    for machine_name, grp in schedule.sort_values("start_time").groupby("machine_name"):
        prev_end = None
        prev_batch = None
        for row in grp.itertuples(index=False):
            if prev_end is not None and row.start_time < prev_end:
                errors.append(f"Machine overlap on {machine_name}: {prev_batch} and {row.batch_id}")
            prev_end = row.end_time
            prev_batch = row.batch_id

    for batch_id, grp in schedule.sort_values("step_number").groupby("batch_id"):
        prev_end = None
        for row in grp.itertuples(index=False):
            if prev_end is not None and row.start_time < prev_end:
                errors.append(f"Step sequencing violation in {batch_id} at step {row.step_number}")
            prev_end = row.end_time

    for row in schedule.itertuples(index=False):
        actual_hours = (row.end_time - row.start_time).total_seconds() / 3600.0
        if abs(actual_hours - row.duration_hours) > 0.0002:
            errors.append(f"Duration mismatch in {row.batch_id} step {row.step_number}")
        windows = machine_windows[row.machine_name]
        in_shift = False
        for start_h, end_h in windows:
            shift_start = pd.Timestamp(row.start_time).normalize() + pd.Timedelta(hours=start_h)
            shift_end = pd.Timestamp(row.start_time).normalize() + pd.Timedelta(hours=end_h)
            if shift_start <= row.start_time and row.end_time <= shift_end:
                in_shift = True
                break
        if not in_shift:
            errors.append(f"Shift boundary violation in {row.batch_id} step {row.step_number}")

    coherence_cols = ["product_item", "product_description", "batch_size", "batch_num"]
    coherence = schedule.groupby("batch_id")[coherence_cols].nunique()
    bad_batches = coherence[(coherence > 1).any(axis=1)].index.tolist()
    for batch_id in bad_batches:
        errors.append(f"Batch coherence violation in {batch_id}")

    if errors:
        raise AssertionError("Schedule validation failed:\n" + "\n".join(errors[:25]))
    return {
        "rows": len(schedule),
        "batches": schedule["batch_id"].nunique(),
        "machines": schedule["machine_name"].nunique(),
        "starts_at": schedule["start_time"].min(),
        "ends_at": schedule["end_time"].max(),
    }


scheduled_operations = build_batch_schedule(df_products, df_steps, df_machines)
validation_summary = validate_schedule(scheduled_operations, df_machines)

scheduled_operations.to_csv("scheduled_production_operations.csv", index=False)
print("Schedule validation passed")
print(validation_summary)
display(scheduled_operations.head(20))


Schedule validation passed
{'rows': 922, 'batches': 152, 'machines': 8, 'starts_at': Timestamp('2026-06-01 08:00:00'), 'ends_at': Timestamp('2026-06-12 13:57:00')}


,batch_id,batch_num,product_item,product_description,dcc_type,batch_size,step_number,step_name,machine_name,cycle_time_seconds,workers_required,start_time,end_time,duration_hours,start_date,end_date,product_batch
0,P001-B001,1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,1,"Wire Cutting & Separation — Single Conductors,...",Kronos EX-580 / Zephyr FX-540,8.40,0.5,2026-06-01 08:00:00.000000000,2026-06-01 08:07:00.000000000,0.116667,2026-06-01,2026-06-01,P001-B001 | Twisted Pair 2xFAKRA Module 32V845...
1,P001-B001,1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,2,"Seal Crimping — FAKRA Contact, Wire Gauge 1.5 AWG",Kronos EX-580 / Zephyr FX-540,5.94,0.5,2026-06-01 08:07:00.000000000,2026-06-01 08:11:57.000000000,0.082500,2026-06-01,2026-06-01,P001-B001 | Twisted Pair 2xFAKRA Module 32V845...
2,P001-B001,1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,3,Manual Connector Build — Coding Cod-C Blue — 9...,Terminal Insertion Station,9.62,1.0,2026-06-01 08:11:57.000000000,2026-06-01 08:19:57.999999999,0.133611,2026-06-01,2026-06-01,P001-B001 | Twisted Pair 2xFAKRA Module 32V845...
3,P001-B001,1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,4,"Cable Bundling & Taping — 15mm PVC Tape, 80mm ...",Pull-Force Testing Station,9.56,0.5,2026-06-01 08:19:57.999999999,2026-06-01 08:27:55.999999999,0.132778,2026-06-01,2026-06-01,P001-B001 | Twisted Pair 2xFAKRA Module 32V845...
4,P001-B001,1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,5,Pin Insertion & Lock — Coding Cod-D Grey — 90°...,Pull-Force Testing Station,11.90,1.0,2026-06-01 08:27:55.999999999,2026-06-01 08:37:50.999999999,0.165278,2026-06-01,2026-06-01,P001-B001 | Twisted Pair 2xFAKRA Module 32V845...
5,P001-B001,1,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,6,"Injection Overmolding — Straight-Exit, 3Q0-973...",Demag ET750 Machine 1-6,15.64,1.0,2026-06-01 08:37:50.999999999,2026-06-01 08:50:52.999999999,0.217222,2026-06-01,2026-06-01,P001-B001 | Twisted Pair 2xFAKRA Module 32V845...
6,P001-B002,2,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,1,"Wire Cutting & Separation — Single Conductors,...",Kronos EX-580 / Zephyr FX-540,8.40,0.5,2026-06-01 08:11:57.000000000,2026-06-01 08:18:57.000000000,0.116667,2026-06-01,2026-06-01,P001-B002 | Twisted Pair 2xFAKRA Module 32V845...
7,P001-B002,2,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,2,"Seal Crimping — FAKRA Contact, Wire Gauge 1.5 AWG",Kronos EX-580 / Zephyr FX-540,5.94,0.5,2026-06-01 08:18:57.000000000,2026-06-01 08:23:54.000000000,0.082500,2026-06-01,2026-06-01,P001-B002 | Twisted Pair 2xFAKRA Module 32V845...
8,P001-B002,2,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,3,Manual Connector Build — Coding Cod-C Blue — 9...,Terminal Insertion Station,9.62,1.0,2026-06-01 08:23:54.000000000,2026-06-01 08:31:54.999999999,0.133611,2026-06-01,2026-06-01,P001-B002 | Twisted Pair 2xFAKRA Module 32V845...
9,P001-B002,2,1,Twisted Pair 2xFAKRA Module 32V845D FA/FB,Dual FAKRA,50,4,"Cable Bundling & Taping — 15mm PVC Tape, 80mm ...",Pull-Force Testing Station,9.56,0.5,2026-06-01 08:37:50.999999999,2026-06-01 08:45:48.999999999,0.132778,2026-06-01,2026-06-01,P001-B002 | Twisted Pair 2xFAKRA Module 32V845...


In [21]:
import plotly.express as px
from IPython.display import clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    widgets = None
    HAS_WIDGETS = False

plot_df = scheduled_operations.copy()
plot_df["display_label"] = "Step " + plot_df["step_number"].astype(str)
plot_df["hover_text"] = (
    plot_df["batch_id"] + "<br>" +
    plot_df["product_description"] + "<br>" +
    plot_df["step_name"]
)

machine_options = sorted(plot_df["machine_name"].unique().tolist())
batch_options = ["All"] + sorted(plot_df["batch_id"].unique().tolist())
min_start = plot_df["start_time"].min().floor("h")
max_end = plot_df["end_time"].max().ceil("h")


def build_schedule_figure(df, selected_machines, start_bound, end_bound):
    fig = px.timeline(
        df,
        x_start="start_time",
        x_end="end_time",
        y="machine_name",
        color="batch_id",
        text="display_label",
        hover_name="hover_text",
        hover_data={
            "machine_name": True,
            "start_time": True,
            "end_time": True,
            "duration_hours": ":.3f",
            "batch_id": True,
            "product_description": True,
            "display_label": False,
            "hover_text": False,
        },
        category_orders={"machine_name": selected_machines},
    )
    fig.update_yaxes(autorange="reversed", title="")
    fig.update_xaxes(range=[start_bound, end_bound], title="", showgrid=True)
    fig.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
    fig.update_layout(
        height=max(360, 72 * max(1, len(selected_machines))),
        margin=dict(l=20, r=20, t=30, b=20),
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(size=13),
        bargap=0.35,
    )
    return fig


if HAS_WIDGETS:
    time_options = [(ts.strftime("%Y-%m-%d %H:%M"), ts) for ts in pd.date_range(min_start, max_end, freq="1h")]

    machine_filter = widgets.SelectMultiple(
        options=machine_options,
        value=tuple(machine_options[: min(4, len(machine_options))]),
        description="Machines",
        rows=min(8, len(machine_options)),
        layout=widgets.Layout(width="360px"),
    )

    batch_filter = widgets.Dropdown(
        options=batch_options,
        value="All",
        description="Batch",
        layout=widgets.Layout(width="360px"),
    )

    range_filter = widgets.SelectionRangeSlider(
        options=time_options,
        index=(0, len(time_options) - 1),
        description="Time",
        layout=widgets.Layout(width="760px"),
    )

    chart_out = widgets.Output()

    def draw_schedule_chart(*_):
        selected_machines = list(machine_filter.value)
        selected_batch = batch_filter.value
        start_bound, end_bound = range_filter.value

        df = plot_df[
            plot_df["machine_name"].isin(selected_machines)
            & (plot_df["end_time"] > start_bound)
            & (plot_df["start_time"] < end_bound)
        ].copy()
        if selected_batch != "All":
            df = df[df["batch_id"] == selected_batch]

        with chart_out:
            clear_output(wait=True)
            if df.empty:
                print("No scheduled operations match the selected filters.")
                return
            build_schedule_figure(df, selected_machines, start_bound, end_bound).show()

    for widget in (machine_filter, batch_filter, range_filter):
        widget.observe(draw_schedule_chart, names="value")

    controls = widgets.VBox([
        widgets.HBox([machine_filter, batch_filter]),
        range_filter,
    ])
    display(controls, chart_out)
    draw_schedule_chart()
else:
    print("ipywidgets is not installed, so showing a static Plotly chart.")
    print("For notebook filters, install it in this kernel with: pip install ipywidgets")
    selected_machines = machine_options[: min(4, len(machine_options))]
    static_df = plot_df[plot_df["machine_name"].isin(selected_machines)].copy()
    build_schedule_figure(static_df, selected_machines, min_start, max_end).show()


ipywidgets is not installed, so showing a static Plotly chart.
For notebook filters, install it in this kernel with: pip install ipywidgets


In [22]:
!pip install nbformat


[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
